## Preparing Images for Feature Detection

# Unit 3: Image Preprocessing for Feature Detection

Welcome back! In our previous units, we built our workbench tools. We learned how to read color images using our `read_color` function and how to inspect them side-by-side using `make_panel`. Now, we are entering **Unit 3**, where we need to prepare our images so that the computer can find matching points between them.

When we stitch images together to make a panorama, our program needs to find common points — called **"features"** — in both images. Feature detectors look for distinct shapes, corners, and edges. They do not care about colors, and they struggle if an image is too dark, washed out, or grainy.

In this lesson, we will build a `preprocess_for_features` function. Our goal is to convert images to grayscale, boost their contrast, and smooth out any distracting noise using OpenCV (`cv2`). In the CodeSignal environment, this library comes pre-installed, so you do not need to worry about setting it up right now.

---

## Fading to Grayscale

Feature detectors process images using complex math. Full-color images have three channels (Blue, Green, and Red), which means three times the amount of data to process. To simplify the math and speed up our program, our first step is always to drop the colors.

Let's start building our preprocessing function by converting the input image to grayscale using OpenCV's `cv2.cvtColor` function:

```python
import cv2

def preprocess_for_features(image):
    # Convert the BGR color image to a single-channel grayscale image
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return gray

```

If you were to check the shape of your image matrix before and after this step, a color image might look like `(900, 1200, 3)`, while the output `gray` image would be just `(900, 1200)`. This simpler format is exactly what our feature detectors expect.

---

## Boosting Local Contrast with CLAHE

Sometimes images are flat or have low contrast, making it difficult to find distinct corners. To fix this, we can boost the contrast.

Instead of boosting the contrast across the whole image (which might wash out bright areas and darken shadows), we use a tool called **CLAHE** (Contrast Limited Adaptive Histogram Equalization).

### How CLAHE Works

To understand CLAHE, we first need to understand two basic concepts:

* **Histogram:** Imagine a bar chart that counts how many pixels of each brightness level exist in an image — from pure black on the left to pure white on the right. This is a histogram. If a photo is "flat" or low-contrast, all the bars will be bunched up in the middle.
* **Histogram Equalization:** This is a technique that takes those bunched-up bars and "stretches" them out across the entire range. This makes the darks darker and the lights lighter, forcing details to become more visible.

CLAHE improves on this with two main principles:

1. **Adaptive (Local):** Instead of looking at the histogram of the entire image (which might over-brighten a sky just to see a dark building), CLAHE divides the image into small tiles (like an 8x8 grid). It calculates and stretches the contrast for each tile individually.
2. **Contrast Limited:** In areas with no detail (like a clear sky), standard equalization can accidentally amplify tiny bits of digital sensor noise until they look like ugly grain. The "clip limit" caps the boost, ensuring we only enhance real features and keep the background clean.

Let's update our function to include optional CLAHE processing. We will use a `clipLimit` of `3.0` to boost contrast and a `tileGridSize` of `8` to divide the image into an 8x8 grid.

```python
import cv2

def preprocess_for_features(
    image,
    use_clahe=True,
    clahe_clip_limit=3.0,
    clahe_tile_grid_size=8,
):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    if use_clahe:
        # Ensure our grid size is at least 1
        tile_size = max(1, int(clahe_tile_grid_size))
        
        # Create the CLAHE tool with our specific settings
        clahe = cv2.createCLAHE(
            clipLimit=clahe_clip_limit,
            tileGridSize=(tile_size, tile_size),
        )
        
        # Apply the contrast boost to our grayscale image
        gray = clahe.apply(gray)
        
    return gray

```

---

## Taming Noise with Gaussian Blur

Cameras sometimes produce grainy images, especially in low light. A feature detector might get confused and think a random speck of digital noise is a valuable corner or edge. To prevent this, we can apply a slight blur to smooth out the image.

### What is a Gaussian Blur?

**Gaussian Blur** is a technique used to reduce image noise and detail. Unlike a simple blur that averages all pixels in a neighborhood equally, a Gaussian blur uses a "weighted average." It follows a Gaussian distribution (a bell-shaped curve), where the pixel at the center has the highest weight, and pixels further away have less influence. This results in a "softer" image that removes high-frequency noise while preserving the overall structure.

### Understanding the Kernel

To apply this blur, OpenCV uses something called a **kernel**. Think of a kernel as a small square window (a matrix) that slides over every pixel in your image:

1. The kernel centers itself over a pixel.
2. It looks at the neighboring pixels within the window.
3. It performs a mathematical calculation (the weighted average) to determine the new value for that center pixel.
4. It moves to the next pixel and repeats the process.

For this math to work correctly, the kernel needs a single, clear center pixel. This is why OpenCV has a strict rule: **the `blur_size` (the width and height of the kernel) must be an odd number**, like 3x3, 5x5, or 7x7. If we pass an even number, OpenCV cannot find a perfect center, and the program will crash. We will add a simple check to automatically add 1 to any even number to keep things safe.

> **Edge Handling:** For pixels on the edges, the kernel window would hang off the image. OpenCV handles this by using border padding — it essentially creates "virtual" pixels outside the boundary (often by reflecting the inner pixels) so the math can still work.

Let's add the blurring logic to our function:

```python
import cv2

def preprocess_for_features(
    image,
    use_clahe=True,
    blur_size=0,
    clahe_clip_limit=3.0,
    clahe_tile_grid_size=8,
):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    if use_clahe:
        tile_size = max(1, int(clahe_tile_grid_size))
        clahe = cv2.createCLAHE(
            clipLimit=clahe_clip_limit,
            tileGridSize=(tile_size, tile_size),
        )
        gray = clahe.apply(gray)
        
    # Only apply blur if a size greater than 0 is requested
    if blur_size > 0:
        # OpenCV requires odd kernel sizes to have a center pixel. 
        # If even, make it odd.
        if blur_size % 2 == 0:
            blur_size += 1
            
        # Apply the Gaussian blur with the specified kernel size
        # (blur_size, blur_size) defines the width and height of the kernel
        gray = cv2.GaussianBlur(gray, (blur_size, blur_size), 0)
        
    return gray

```

---

## Summary & Practice Prep

We now have a complete, robust `preprocess_for_features` function! It takes an image, strips away the distracting colors, boosts the hidden details, and smooths out the noisy grain.

To see how this fits into the bigger picture, here is a quick look at how you might use this new function alongside the tools we built in previous units:

```python
# Assuming read_color and make_panel are imported from our toolkit
image = read_color("photo.jpg")

# Prepare the image using our new function
prepared = preprocess_for_features(
    image,
    use_clahe=True,
    blur_size=3
)

# Display the original color image and the prepared grayscale image side-by-side
panel = make_panel(
    [
        ("original", image),
        ("feature input", prepared),
    ],
    max_size=900,
)

cv2.imshow("preprocessing", panel)
cv2.waitKey(0)
cv2.destroyAllWindows()

```

When you run code like this, a window will pop up showing your original color image on the left and a crisp, high-contrast grayscale version on the right — ready for feature detection!

Now, it is your turn. In the upcoming practice exercises, you will write this preprocessing logic yourself and integrate it into our growing stitching workbench. Let's jump into the code!

## Boosting Contrast for Sharper Features

With the theory behind grayscale conversion and CLAHE fresh in mind, it is time to turn that knowledge into working code inside cvkit.py.

Your job is to finish the preprocess_for_features function so that it can boost local contrast with CLAHE. The grayscale conversion has already been completed for you, so you only need to fill in the three TODO lines inside the if use_clahe: block.

Here is what each TODO requires:

    Clamp clahe_tile_grid_size so that it is never smaller than 1, and save it to a variable named tile_size.
    Build the CLAHE tool by calling cv2.createCLAHE with clipLimit=clahe_clip_limit and tileGridSize=(tile_size, tile_size).
    Apply that tool to gray and store the result back in gray.

This is the first real building block of your stitching pipeline, so take your time and enjoy seeing the contrast pop.

```
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output


def make_panel(items, max_size=900):
    if not items:
        raise ValueError("make_panel requires at least one image")

    tiles = []
    for name, image in items:
        labeled = label_image(image, name)
        tiles.append(resize_long_edge(labeled, max_size=max_size))

    max_height = max(tile.shape[0] for tile in tiles)
    tiles = [pad_to_height(tile, max_height) for tile in tiles]
    return np.hstack(tiles)


def preprocess_for_features(
    image,
    use_clahe=True,
    clahe_clip_limit=3.0,
    clahe_tile_grid_size=8,
):
    # Convert the BGR color image to a single-channel grayscale image
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if use_clahe:
        # TODO: Clamp the tile size to be at least 1 (use max(1, int(...)))
        #       and store the result in a variable called tile_size

        # TODO: Create the CLAHE tool by calling cv2.createCLAHE with
        #       clipLimit=clahe_clip_limit and tileGridSize=(tile_size, tile_size)

        # TODO: Apply the CLAHE tool to the gray image and reassign the result to gray

        pass

    return gray

```

Here is the completed `preprocess_for_features` function with the CLAHE contrast boosting logic implemented in `cvkit.py`:

```python
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output


def make_panel(items, max_size=900):
    if not items:
        raise ValueError("make_panel requires at least one image")

    tiles = []
    for name, image in items:
        labeled = label_image(image, name)
        tiles.append(resize_long_edge(labeled, max_size=max_size))

    max_height = max(tile.shape[0] for tile in tiles)
    tiles = [pad_to_height(tile, max_height) for tile in tiles]
    return np.hstack(tiles)


def preprocess_for_features(
    image,
    use_clahe=True,
    clahe_clip_limit=3.0,
    clahe_tile_grid_size=8,
):
    # Convert the BGR color image to a single-channel grayscale image
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if use_clahe:
        # Clamp the tile size to be at least 1
        tile_size = max(1, int(clahe_tile_grid_size))

        # Create the CLAHE tool with specified parameters
        clahe = cv2.createCLAHE(
            clipLimit=clahe_clip_limit,
            tileGridSize=(tile_size, tile_size),
        )

        # Apply the CLAHE contrast boost to the grayscale image
        gray = clahe.apply(gray)

    return gray

```

## Comparing CLAHE Parameter Sweeps

Nice work wiring up CLAHE into the preprocessing pipeline! Now it's time to see how its two main parameters change the look of an image side by side.

Open solution.py and complete the four TODO blocks to build two comparison panels:

    Fill clip_limits with a few values to try (for example, 1.0, 3.0, 6.0, 10.0).
    Loop through clip_limits, call preprocess_for_features with each value passed as clahe_clip_limit, and append (f"clip={clip}", prepared) to clip_items.
    Fill tile_grid_sizes with a few values (for example, 2, 4, 8, 16).
    Loop through tile_grid_sizes, call preprocess_for_features with each value passed as clahe_tile_grid_size, and append (f"tile={tile}", prepared) to tile_items.

When running the task ensure to compare the images in our image server and click continue to continue the execution — this is the fastest way to build intuition for what each CLAHE knob actually does.

```
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    args = parser.parse_args()

    image = read_color(args.path)

    # TODO: Define a list of clip limit values to compare
    #       (try something like 1.0, 3.0, 6.0, 10.0)
    clip_limits = []

    # TODO: Loop through clip_limits and call preprocess_for_features for each
    #       value, passing it as clahe_clip_limit. Append a tuple of
    #       (f"clip={clip}", prepared) to clip_items.
    clip_items = []

    # TODO: Define a list of tile grid size values to compare
    #       (try something like 2, 4, 8, 16)
    tile_grid_sizes = []

    # TODO: Loop through tile_grid_sizes and call preprocess_for_features for
    #       each value, passing it as clahe_tile_grid_size. Append a tuple of
    #       (f"tile={tile}", prepared) to tile_items.
    tile_items = []

    clip_panel = make_panel(clip_items, max_size=900)
    tile_panel = make_panel(tile_items, max_size=900)

    cv2.imshow("clip limit comparison", clip_panel)
    cv2.imshow("tile grid size comparison", tile_panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

Here is the completed implementation for `solution.py` to compare CLAHE parameter sweeps across different clip limits and tile grid sizes:

```python
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    args = parser.parse_args()

    image = read_color(args.path)

    # Define a list of clip limit values to compare
    clip_limits = [1.0, 3.0, 6.0, 10.0]

    # Loop through clip_limits and call preprocess_for_features for each value
    clip_items = []
    for clip in clip_limits:
        prepared = preprocess_for_features(image, clahe_clip_limit=clip)
        clip_items.append((f"clip={clip}", prepared))

    # Define a list of tile grid size values to compare
    tile_grid_sizes = [2, 4, 8, 16]

    # Loop through tile_grid_sizes and call preprocess_for_features for each value
    tile_items = []
    for tile in tile_grid_sizes:
        prepared = preprocess_for_features(image, clahe_tile_grid_size=tile)
        tile_items.append((f"tile={tile}", prepared))

    clip_panel = make_panel(clip_items, max_size=900)
    tile_panel = make_panel(tile_items, max_size=900)

    cv2.imshow("clip limit comparison", clip_panel)
    cv2.imshow("tile grid size comparison", tile_panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Smoothing Out the Noisy Grain

With CLAHE already boosting contrast nicely, it is time to add the final smoothing step that keeps feature detectors from chasing noise.

In this exercise, you will finish preprocess_for_features by adding the optional Gaussian blur stage right after the CLAHE block. Follow the TODO comments to wire it up:

    Wrap the blur logic in a check so that it only runs when blur_size is greater than 0.
    Inside that check, bump blur_size up by 1 if it is even, since OpenCV needs an odd kernel.
    Then, call cv2.GaussianBlur with a (blur_size, blur_size) kernel and a sigma of 0, and reassign the result back to gray.

Once this is in place, your preprocessing pipeline will be complete and ready to feed clean images to a feature detector.

```
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output


def make_panel(items, max_size=900):
    if not items:
        raise ValueError("make_panel requires at least one image")

    tiles = []
    for name, image in items:
        labeled = label_image(image, name)
        tiles.append(resize_long_edge(labeled, max_size=max_size))

    max_height = max(tile.shape[0] for tile in tiles)
    tiles = [pad_to_height(tile, max_height) for tile in tiles]
    return np.hstack(tiles)


def preprocess_for_features(
    image,
    use_clahe=True,
    blur_size=0,
    clahe_clip_limit=3.0,
    clahe_tile_grid_size=8,
):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if use_clahe:
        tile_size = max(1, int(clahe_tile_grid_size))
        clahe = cv2.createCLAHE(
            clipLimit=clahe_clip_limit,
            tileGridSize=(tile_size, tile_size),
        )
        gray = clahe.apply(gray)

    # TODO: Only run the blur step when blur_size is greater than 0, so the
    #       blur stays optional

    # TODO: Inside that check, if blur_size is even, add 1 to it so OpenCV
    #       gets an odd kernel size with a clear center pixel

    # TODO: Still inside the check, call cv2.GaussianBlur with a
    #       (blur_size, blur_size) kernel and a sigma of 0, then reassign
    #       the result back to gray

    return gray

```

Here is the complete implementation for `cvkit.py` with the optional Gaussian blur stage added to `preprocess_for_features`:

```python
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output


def make_panel(items, max_size=900):
    if not items:
        raise ValueError("make_panel requires at least one image")

    tiles = []
    for name, image in items:
        labeled = label_image(image, name)
        tiles.append(resize_long_edge(labeled, max_size=max_size))

    max_height = max(tile.shape[0] for tile in tiles)
    tiles = [pad_to_height(tile, max_height) for tile in tiles]
    return np.hstack(tiles)


def preprocess_for_features(
    image,
    use_clahe=True,
    blur_size=0,
    clahe_clip_limit=3.0,
    clahe_tile_grid_size=8,
):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if use_clahe:
        tile_size = max(1, int(clahe_tile_grid_size))
        clahe = cv2.createCLAHE(
            clipLimit=clahe_clip_limit,
            tileGridSize=(tile_size, tile_size),
        )
        gray = clahe.apply(gray)

    # Only run the blur step when blur_size is greater than 0
    if blur_size > 0:
        # Ensure the kernel size is odd for OpenCV
        if blur_size % 2 == 0:
            blur_size += 1

        # Apply Gaussian blur with the odd kernel size and sigma=0
        gray = cv2.GaussianBlur(gray, (blur_size, blur_size), 0)

    return gray

```

## Comparing Blur Strengths Side by Side

Skip to main content
Preparing Images for Feature Detection

Now that the preprocessing pipeline is complete with the Gaussian blur stage, it is time to see how that blur knob affects your images.

In solution.py, you will build a side-by-side panel that compares different blur sizes on the same picture, just like the CLAHE sweep you performed earlier.

Here is what to do:

    Fill in blur_sizes with a few values to compare (try 0, 3, 7, 15 to cover everything from "no blur" to "pretty blurry").
    Loop through blur_sizes and call preprocess_for_features for each, passing the value as blur_size.
    Append a tuple of (f"blur={blur}", prepared) to blur_items so that make_panel can label each tile.

Once it runs, you will gain a clear visual sense of how much smoothing is "just right" before feature detection begins.

When running the task, remember to click continue in the image preview window when you inspected the image generated. If you don't do it, the program will be stuck in the cv2.waitKey(0) line and never end!

```
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    args = parser.parse_args()

    image = read_color(args.path)

    # TODO: Define a list of blur sizes to compare
    #       (try something like 0, 3, 7, 15)
    blur_sizes = []

    # TODO: Loop through blur_sizes and call preprocess_for_features for each
    #       value, passing it as blur_size. Append a tuple of
    #       (f"blur={blur}", prepared) to blur_items.
    blur_items = []

    blur_panel = make_panel(blur_items, max_size=900)

    cv2.imshow("blur size comparison", blur_panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

Here is the completed implementation for `solution.py` to compare different Gaussian blur kernel sizes side by side:

```python
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    args = parser.parse_args()

    image = read_color(args.path)

    # Define a list of blur sizes to compare
    blur_sizes = [0, 3, 7, 15]

    # Loop through blur_sizes and call preprocess_for_features for each value
    blur_items = []
    for blur in blur_sizes:
        prepared = preprocess_for_features(image, blur_size=blur)
        blur_items.append((f"blur={blur}", prepared))

    blur_panel = make_panel(blur_items, max_size=900)

    cv2.imshow("blur size comparison", blur_panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```